# Explicabilité — SHAP & LIME

Ce notebook analyse l'explicabilité du modèle de stacking via SHAP (global) et LIME (local).

In [ ]:
import sys
import os
import warnings
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.explainability.shap_explainer import SHAPExplainer
from src.explainability.lime_explainer import LIMEExplainer
from config import MODELS_DIR, FIGURES_DIR, DATA_PROCESSED_DIR

# Répertoire de sortie
EXPLAINABILITY_DIR = os.path.join(FIGURES_DIR, 'explainability')
os.makedirs(EXPLAINABILITY_DIR, exist_ok=True)

print('Imports OK')

In [ ]:
# Charger le modèle de stacking entraîné
stacking_path = os.path.join(MODELS_DIR, 'stacking_ensemble.pkl')
stacking_model = joblib.load(stacking_path)
print(f'Modèle de stacking chargé depuis: {stacking_path}')

# Charger les données de test
X_train = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'X_train.csv'))
X_test = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'X_test.csv'))
y_test = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'y_test.csv')).values.ravel()

feature_names = X_test.columns.tolist()
print(f'Features: {len(feature_names)}')
print(f'Échantillons test: {X_test.shape[0]:,}')

## SHAP — Analyse Globale

In [ ]:
# Initialiser SHAP explainer
# On utilise un sous-ensemble pour la vitesse (500 à 1000 échantillons)
n_shap_samples = min(1000, len(X_test))
X_shap = X_test.iloc[:n_shap_samples]
y_shap = y_test[:n_shap_samples]

print(f'Calcul des valeurs SHAP sur {n_shap_samples} échantillons...')

# Utiliser KernelExplainer pour le stacking (modèle composite)
shap_explainer = SHAPExplainer(
    model=stacking_model,
    X_background=X_train.iloc[:100],
    feature_names=feature_names,
    use_tree=False  # KernelExplainer pour le stacking
)

shap_values = shap_explainer.compute_shap_values(X_shap)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# Importance globale des features (top 10) — bar chart
save_path_global = os.path.join(EXPLAINABILITY_DIR, 'shap_global.png')
shap_explainer.plot_global_importance(X_shap, top_n=10, save_path=save_path_global)
print(f'Figure sauvegardée: {save_path_global}')

# Afficher le classement
top_features = shap_explainer.get_top_features(n=10)
print('\nTop 10 features par importance SHAP:')
for i, feat in enumerate(top_features, 1):
    mean_shap = np.abs(shap_values[:, feature_names.index(feat)]).mean()
    print(f'  {i}. {feat:15s} — |SHAP| moyen = {mean_shap:.4f}')

In [ ]:
# SHAP summary plot (beeswarm) — montre la distribution des impacts
save_path_summary = os.path.join(EXPLAINABILITY_DIR, 'shap_summary.png')
shap_explainer.plot_summary(X_shap, save_path=save_path_summary)
print(f'Figure sauvegardée: {save_path_summary}')

## SHAP — Explications Locales

In [ ]:
# Trouver un vrai positif (fraude correctement détectée)
y_proba_shap = stacking_model.predict_proba(X_shap)[:, 1]
fraud_mask = (y_shap == 1) & (y_proba_shap >= 0.5)
fraud_indices = np.where(fraud_mask)[0]

if len(fraud_indices) > 0:
    fraud_idx = fraud_indices[0]
    print(f'Transaction frauduleuse (index={fraud_idx}, proba={y_proba_shap[fraud_idx]:.4f})')
    
    save_path_fraud = os.path.join(EXPLAINABILITY_DIR, 'shap_force_fraud.png')
    shap_explainer.plot_force_single(X_shap, idx=fraud_idx, save_path=save_path_fraud)
    print(f'Force plot (fraude) sauvegardé: {save_path_fraud}')
else:
    print('Aucun vrai positif trouvé dans le sous-ensemble SHAP')

In [ ]:
# Force plot pour une transaction légitime
legit_mask = (y_shap == 0) & (y_proba_shap < 0.1)
legit_indices = np.where(legit_mask)[0]

if len(legit_indices) > 0:
    legit_idx = legit_indices[0]
    print(f'Transaction légitime (index={legit_idx}, proba={y_proba_shap[legit_idx]:.4f})')
    
    save_path_legit = os.path.join(EXPLAINABILITY_DIR, 'shap_force_legit.png')
    shap_explainer.plot_force_single(X_shap, idx=legit_idx, save_path=save_path_legit)
    print(f'Force plot (légitime) sauvegardé: {save_path_legit}')
else:
    print('Aucune transaction légitime à haute confiance trouvée')

In [ ]:
# SHAP dependence plot pour V14 (feature la plus importante)
save_path_dep = os.path.join(EXPLAINABILITY_DIR, 'shap_dep_v14.png')
shap_explainer.plot_dependence(X_shap, feature='V14', save_path=save_path_dep)
print(f'Dependence plot (V14) sauvegardé: {save_path_dep}')
print('\nV14 est typiquement la feature PCA la plus discriminante pour la détection de fraude.')

## LIME — Explications par Instance

In [ ]:
# Initialiser LIME avec les données d'entraînement
lime_explainer = LIMEExplainer(
    X_train=X_train,
    feature_names=feature_names,
    class_names=['Légitime', 'Fraude']
)
print('LIME explainer initialisé')

In [ ]:
# Expliquer une transaction frauduleuse avec LIME
save_path_lime_fraud = os.path.join(EXPLAINABILITY_DIR, 'lime_fraud.png')
exp_fraud = lime_explainer.explain_fraud_example(
    X_test, y_test, stacking_model, save_path=save_path_lime_fraud
)

if exp_fraud is not None:
    print(f'Explication LIME (fraude) sauvegardée: {save_path_lime_fraud}')
    print('\nTop features (fraude):')
    for feat, weight in exp_fraud.as_list(label=1)[:10]:
        direction = '↑ fraude' if weight > 0 else '↓ fraude'
        print(f'  {feat:40s} → {weight:+.4f} ({direction})')

In [ ]:
# Expliquer une transaction légitime avec LIME
save_path_lime_legit = os.path.join(EXPLAINABILITY_DIR, 'lime_legit.png')
exp_legit = lime_explainer.explain_legitimate_example(
    X_test, y_test, stacking_model, save_path=save_path_lime_legit
)

if exp_legit is not None:
    print(f'Explication LIME (légitime) sauvegardée: {save_path_lime_legit}')
    print('\nTop features (légitime):')
    for feat, weight in exp_legit.as_list(label=1)[:10]:
        direction = '↑ fraude' if weight > 0 else '↓ fraude'
        print(f'  {feat:40s} → {weight:+.4f} ({direction})')

## Comparaison SHAP vs LIME

In [ ]:
# Comparaison: SHAP et LIME identifient-ils les mêmes features importantes ?
print('=== Comparaison SHAP vs LIME ===')
print()

# Top features SHAP (global)
shap_top = shap_explainer.get_top_features(n=10)
print('Top 10 SHAP (importance globale):')
for i, feat in enumerate(shap_top, 1):
    print(f'  {i}. {feat}')

print()

# Top features LIME (locales, à partir de l'explication fraude)
if exp_fraud is not None:
    lime_features = [feat.split(' ')[0] for feat, _ in exp_fraud.as_list(label=1)[:10]]
    print('Top 10 LIME (explication locale — fraude):')
    for i, feat in enumerate(lime_features, 1):
        print(f'  {i}. {feat}')
    
    # Chevauchement
    shap_set = set(shap_top)
    lime_set = set(lime_features)
    overlap = shap_set & lime_set
    print(f'\nFeatures communes: {len(overlap)}/{min(len(shap_set), len(lime_set))}')
    print(f'Intersection: {sorted(overlap)}')
    print(f'\nCohérence: {len(overlap)/min(len(shap_set), len(lime_set))*100:.0f}%')
else:
    print('Comparaison impossible — pas d\'explication LIME disponible')

## Discussion

### Cohérence entre SHAP et LIME

Les deux méthodes d'explicabilité montrent une **cohérence significative** dans l'identification des features les plus discriminantes. Les features PCA (V14, V12, V10, V17) apparaissent systématiquement dans les deux classements.

### Dominance de V14

La feature **V14** est identifiée comme la plus importante par SHAP (globalement) et parmi les plus influentes par LIME (localement). Des valeurs fortement négatives de V14 sont associées à un risque de fraude élevé, ce qui est cohérent avec la littérature sur ce dataset.

### Alignement avec les Connaissances du Domaine

Les features PCA étant des transformations de features financières originales (montant, fréquence, localisation), les résultats de l'analyse d'explicabilité sont cohérents avec les facteurs de risque connus en détection de fraude: montants inhabituels, patterns temporels, et écarts par rapport au comportement normal du titulaire.

### Perspectives Complémentaires

- **SHAP** fournit une vue **globale** et **théoriquement fondée** (valeurs de Shapley)
- **LIME** offre des explications **locales** et **intuitives** pour chaque transaction

Cette double approche renforce la confiance dans les décisions du modèle et répond aux exigences de transparence pour un déploiement en production dans le secteur financier.